# Choose a GNM cutoff

Search a finite grid using experimental B-factor correlation. The routine
keeps the selected nodes, skips degenerate/constant-theoretical candidates
and refits the winner. If none is admissible it raises
`CutoffOptimizationError` and preserves the previous state.

In [ ]:
from importlib.resources import as_file, files
import numpy as np
import molsysmt as msm
from elastnetmt import AnisotropicNetworkModel, GaussianNetworkModel, pyunitwizard as puw

resource = files("molsysmt").joinpath("data/pdb/1tcd.pdb")
with as_file(resource) as path:
    molecular_system = msm.convert(str(path), to_form="molsysmt.MolSys")

In [ ]:
gnm = GaussianNetworkModel(molecular_system, engine="vectorized")
cutoff, correlation = gnm.get_best_cutoff(
    min_cutoff="6 angstroms", max_cutoff="10 angstroms", steps=5,
)
assert np.isfinite(correlation)
assert 6 <= puw.get_value(cutoff, to_unit="angstroms") <= 10
assert gnm.b_factors_exp is not None
fresh = GaussianNetworkModel(molecular_system, cutoff=cutoff, engine="vectorized")
fresh_scale, fresh_correlation = fresh.fit_to_experimental_b_factors()
np.testing.assert_allclose(gnm.get_b_factors(), fresh.get_b_factors())
np.testing.assert_allclose(correlation, fresh_correlation)
print(f"Best grid cutoff: {cutoff}; correlation: {correlation:.3f}")

This score is a calibration criterion for the supplied structure and profile;
it does not establish a universally optimal physical cutoff. Invalid
experimental data or a genuine numerical/backend failure propagate.